# <font color="blue">**WORD2VEC VARIANT USING NUMPY**</font>
## Author: Virginia Antón Yuste

**Task Description**: Implement the core training loop of word2vec in pure NumPy (no PyTorch / TensorFlow or other ML frameworks). The applicant is free to choose any suitable text dataset. The task is to implement the optimization procedure (forward pass, loss, gradients, and parameter updates) for a standard word2vec variant (e.g. skip-gram with negative sampling or CBOW).


This project uses the *Bedtime Stories* dataset available on Hugging Face. The dataset contains a collection of short bedtime stories written in English, written for children. Each entry includes a full narrative text with simple vocabulary and clear sentence structures.

In [3]:
import numpy as np
import pandas as pd
import re
import pickle # to save and load the model
import os # to join the path
import random # for randomly generated numbers
import time # to record time for the report
import matplotlib # to plot the loss of experiments for the report

In [5]:
df = pd.read_csv("hf://datasets/gofilipa/bedtime_stories/stories.csv")

In [6]:
df.head()

,stories
0,The stars twinkled in the night sky as little ...
1,Once upon a time there was a little girl named...
2,Alice was so excited for her first sleepover a...
3,"Once upon a time, there lived two best friends..."
4,Max loved the stars. Every night he would lay ...


In [7]:
df.shape

(199, 1)

## 1. DATA PREPARATION

As it is a DataFrame and we want to work with plain text we have to first convert it. 

In [13]:
corpus = " ".join(df["stories"].astype(str))

In [15]:
print(corpus)

The stars twinkled in the night sky as little Eva lay in her bed dreaming of adventures. She imagined flying on the back of a dragon across the clouds, soaring over the tallest mountains. Just then, Eva heard a gentle whisper that seemed to come from everywhere. It said, "It's time for beddy-byes now, dream sweetly little one." Eva smiled and snuggled into her cozy blankets, dreaming of more magical places. Once upon a time there was a little girl named Sam. Every night she snuggled into her warm bed for a cozy sleep. One night, while snuggling with her teddy bear, Sam heard a gentle voice coming from the window. She got up to investigate and saw a soft, glowing light outside. It was a fairy with a big smile who asked Sam to join her on a magical adventure. They flew around the night sky, visiting different planets and galaxies. When the sun began to rise they returned back to Sam's window. The magical fairy gave Sam a hug and Sam went back to bed filled with wonderful memories. Alice 

Then we tokenize the data in order to feed our model. The first step, when it comes to NLP tasks, is to tokenize the text (divide in small units, such as words). In this step we are also going to remove the punctuations ansd so on.  

In [18]:
def tokenize(text):
    text=text.lower()
    tokens= re.findall(r'[a-zA-Z]+[\w^\']*|[a-zA-Z]+[\w^\']*', text)
    return tokens

In [22]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

# al tokenizar, filtrar las stopwords
tokens= tokenize(corpus)
tokens = [w for w in tokens if w not in stop_words]
print(tokens)

['stars', 'twinkled', 'night', 'sky', 'little', 'eva', 'lay', 'bed', 'dreaming', 'adventures', 'imagined', 'flying', 'back', 'dragon', 'across', 'clouds', 'soaring', 'tallest', 'mountains', 'eva', 'heard', 'gentle', 'whisper', 'seemed', 'come', 'everywhere', 'said', 'time', 'beddy', 'byes', 'dream', 'sweetly', 'little', 'one', 'eva', 'smiled', 'snuggled', 'cozy', 'blankets', 'dreaming', 'magical', 'places', 'upon', 'time', 'little', 'girl', 'named', 'sam', 'every', 'night', 'snuggled', 'warm', 'bed', 'cozy', 'sleep', 'one', 'night', 'snuggling', 'teddy', 'bear', 'sam', 'heard', 'gentle', 'voice', 'coming', 'window', 'got', 'investigate', 'saw', 'soft', 'glowing', 'light', 'outside', 'fairy', 'big', 'smile', 'asked', 'sam', 'join', 'magical', 'adventure', 'flew', 'around', 'night', 'sky', 'visiting', 'different', 'planets', 'galaxies', 'sun', 'began', 'rise', 'returned', 'back', "sam's", 'window', 'magical', 'fairy', 'gave', 'sam', 'hug', 'sam', 'went', 'back', 'bed', 'filled', 'wonderf

Let's see how many words in total do we have. 

In [25]:
len(tokens)

9701

For this task I keep the words forms and the stopwords. 

Now we are going to map between tokens and indices, and viceversa. For vector representation. 

In [29]:
def mapTokens(tks):
    wordToId= {}
    idToWord = {}
    for idx, token in enumerate(set(tks)):
        wordToId[token] = idx
        idToWord[idx]=token
    return wordToId, idToWord
    

In [31]:
wordToId, idToWord = mapTokens(tokens)
wordToId

{'stumbled': 0,
 'adventures': 1,
 'yawning': 2,
 'mug': 3,
 'bags': 4,
 'sarah': 5,
 "tomorrow's": 6,
 'mushroom': 7,
 'wherever': 8,
 'noise': 9,
 'despite': 10,
 'byes': 11,
 'peak': 12,
 'fields': 13,
 'least': 14,
 'magic': 15,
 'onwards': 16,
 'wands': 17,
 'uncle': 18,
 'susie': 19,
 'lovable': 20,
 'sparkled': 21,
 'ended': 22,
 'enjoying': 23,
 'join': 24,
 "unicorn's": 25,
 'back': 26,
 'around': 27,
 'long': 28,
 'daisy': 29,
 'dragons': 30,
 'courage': 31,
 'cabin': 32,
 'pink': 33,
 'hugged': 34,
 'laid': 35,
 'imaginable': 36,
 'lily': 37,
 'john': 38,
 'perched': 39,
 "sammy's": 40,
 'style': 41,
 'constantly': 42,
 'rocks': 43,
 'finished': 44,
 'joe': 45,
 'tell': 46,
 'awaited': 47,
 'adventure': 48,
 'clover': 49,
 'grown': 50,
 'toe': 51,
 'stick': 52,
 'take': 53,
 'sent': 54,
 'voice': 55,
 'comets': 56,
 'routine': 57,
 'wished': 58,
 'strange': 59,
 'happiness': 60,
 'knew': 61,
 'benny': 62,
 'swimming': 63,
 'beside': 64,
 'shake': 65,
 'saying': 66,
 'aliens'

In [33]:
#Now we build the indexed vocabulary
indexed_corpus = [wordToId[word] for word in tokens]

In [35]:
print(indexed_corpus[:10])

[695, 358, 305, 1260, 741, 1304, 1356, 956, 320, 1]


**Now we generate the sample pairs:**

In [38]:
def pairGeneration(corpus, window):
    pairs=[]
    for i,target in enumerate(corpus):
        start = max(0, i - window)
        end = min(len(corpus), i + window + 1)
        for j in range(start,end):
            if i!=j:
                context = corpus[j]
                pairs.append((target, context))
    return pairs
    

In [40]:
window_size = 2
pairs = pairGeneration(indexed_corpus, window_size)

In [42]:
print(pairs[:10]) #(target_id, context_id)

[(695, 358), (695, 305), (358, 695), (358, 305), (358, 1260), (305, 695), (305, 358), (305, 1260), (305, 741), (1260, 358)]


We finish the first task! We save the result for the next notebook

In [45]:
import pickle

with open("word2id.pkl", "wb") as f:
    pickle.dump(wordToId, f)

with open("id2word.pkl", "wb") as f:
    pickle.dump(idToWord, f)

with open("corpus.pkl", "wb") as f:
    pickle.dump(indexed_corpus, f)

with open("pairs.pkl", "wb") as f:
    pickle.dump(pairs, f)